In [2]:
import pandas as pd
import os
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [3]:
# PROCESS SINGLE SUBJECT
#=========================

def process_subject_folder(subject_folder, interval=30):
    files = os.listdir(subject_folder)

    base_file = next((f for f in files if "_base.csv" in f), None)
    after_lunch_file = next((f for f in files if "_after_lunch.csv" in f), None)
    stress_file = next((f for f in files if "_stress_induced.csv" in f), None)

    if not all([base_file, after_lunch_file, stress_file]):
        print(f"Missing files in {subject_folder}")
        return None

    # Load CSV files
    base_df = pd.read_csv(os.path.join(subject_folder, base_file))
    after_lunch_df = pd.read_csv(os.path.join(subject_folder, after_lunch_file))
    stress_df = pd.read_csv(os.path.join(subject_folder, stress_file))

    # Remove missing values
    for df in [base_df, after_lunch_df, stress_df]:
        df.dropna(subset=['Heart Rate Ear(BPM)', 'GSR', 'Object Temperature(F)'], inplace=True)

    # Fix Timestamp format
    for df in [base_df, after_lunch_df, stress_df]:
        df['Timestamp'] = df['Timestamp'].apply(
            lambda x: f"{x}:00" if len(str(x).split(":")) == 2 else str(x)
        )
        df['Timestamp'] = pd.to_timedelta(df['Timestamp'])

    # Resample
    def resample_df(df):
        df_numeric = df.select_dtypes(include=np.number).copy()
        df_numeric.index = df['Timestamp']
        df_resampled = df_numeric.resample(f'{interval}s').mean()
        df_resampled.dropna(inplace=True)
        return df_resampled.round(3)

    base_df = resample_df(base_df)
    after_df = resample_df(after_lunch_df)
    stress_df = resample_df(stress_df)

    # Baseline mean
    base_hr_mean = base_df['Heart Rate Ear(BPM)'].mean()
    base_gsr_mean = base_df['GSR'].mean()
    base_temp_mean = base_df['Object Temperature(F)'].mean()

    # Compute indices
    def compute_indices_with_qi(df):
        weighted_list = []
        qi_list = []

        for _, row in df.iterrows():
            # Weighted Stress
            hr_n = row['Heart Rate Ear(BPM)'] / 100
            gsr_n = row['GSR'] / 1000
            temp_n = row['Object Temperature(F)'] / 100

            weighted = 0.35 * hr_n + 0.45 * gsr_n + 0.20 * temp_n
            weighted_list.append(weighted)

            # Baseline relative
            hr_rel = (row['Heart Rate Ear(BPM)'] - base_hr_mean) / base_hr_mean
            gsr_rel = (row['GSR'] - base_gsr_mean) / base_gsr_mean
            temp_rel = (row['Object Temperature(F)'] - base_temp_mean) / base_temp_mean

            # Normalize 0–1
            hr_norm = np.clip(hr_rel + 0.5, 0, 1)
            gsr_norm = np.clip(gsr_rel + 0.5, 0, 1)
            temp_norm = np.clip(temp_rel + 0.5, 0, 1)

            qi = np.exp(0.35 * hr_norm + 0.45 * gsr_norm + 0.20 * temp_norm)
            qi_list.append(qi)

        df['Weighted_Stress'] = weighted_list
        df['QuantityIndex_Stress'] = qi_list

        # Classification
        mean_qi = np.mean(qi_list)
        std_qi = np.std(qi_list)

        def classify(x):
            if x < mean_qi - std_qi:
                return "Low"
            elif x > mean_qi + std_qi:
                return "High"
            else:
                return "Medium"

        df['StressIndex_Level'] = df['QuantityIndex_Stress'].apply(classify)

        return df

    # Apply
    base_df = compute_indices_with_qi(base_df)
    after_df = compute_indices_with_qi(after_df)
    stress_df = compute_indices_with_qi(stress_df)

    # Combine
    final_df = pd.concat([
        base_df.assign(Condition='Base'),
        after_df.assign(Condition='After_Lunch'),
        stress_df.assign(Condition='Stress_Induced')
    ])

    final_df = final_df.reset_index(drop=True)

    return final_df


In [4]:
# PROCESS ALL SUBJECTS
#=========================

def process_all_subjects(main_folder, output_folder, interval=30):
    os.makedirs(output_folder, exist_ok=True)

    all_data = []

    subjects = [d for d in os.listdir(main_folder) if os.path.isdir(os.path.join(main_folder, d))]

    for subj in subjects:
        print(f"Processing {subj}...")
        subj_path = os.path.join(main_folder, subj)

        df = process_subject_folder(subj_path, interval)
        if df is None:
            continue

        all_data.append(df)

        save_path = os.path.join(output_folder, f"{subj}_final_stress.csv")
        df.to_csv(save_path, index=False)

    print("All subjects processed!")

    return pd.concat(all_data, ignore_index=True)


In [5]:
# ML PART (IMPORTANT)
#=========================

def train_linear_model(final_df):
    X = final_df[['Heart Rate Ear(BPM)', 'GSR', 'Object Temperature(F)']]
    y = final_df['QuantityIndex_Stress']

    # Normalize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Train model
    model = LinearRegression()
    model.fit(X_scaled, y)

    # Prediction (fit line)
    y_pred = model.predict(X_scaled)

    # R²
    r2 = r2_score(y, y_pred)

    print("\n=== MODEL RESULT ===")
    print("Weights (scaled):", model.coef_)
    print("Intercept:", model.intercept_)
    print("R² Score:", r2)

    # Raw equation
    means = scaler.mean_
    stds = scaler.scale_

    w_raw = model.coef_ / stds
    b_raw = model.intercept_ - np.sum((model.coef_ * means) / stds)

    print("\n=== FINAL EQUATION (RAW DATA) ===")
    print(f"Stress = {w_raw[0]}*HR + {w_raw[1]}*GSR + {w_raw[2]}*Temp + {b_raw}")

    return model, scaler, r2


In [6]:
# RUN
#=========================

main_folder = r"D:\MONITOR(DEMO)"
output_folder = r"D:\MONITOR(DEMO)\Final"

final_combined_df = process_all_subjects(main_folder, output_folder, interval=30)

# Train model
model, scaler, r2 = train_linear_model(final_combined_df)

Processing .idea...
Missing files in D:\MONITOR(DEMO)\.idea
Processing C01...


C:\Users\DELL\AppData\Local\Temp\ipykernel_18052\1648230092.py:37: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  return df_resampled.round(3)


Processing C02...


C:\Users\DELL\AppData\Local\Temp\ipykernel_18052\1648230092.py:37: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  return df_resampled.round(3)


Processing C03...


...

Processing C26...


C:\Users\DELL\AppData\Local\Temp\ipykernel_18052\1648230092.py:37: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  return df_resampled.round(3)


Processing C27...


C:\Users\DELL\AppData\Local\Temp\ipykernel_18052\1648230092.py:37: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  return df_resampled.round(3)


Processing Final...
Missing files in D:\MONITOR(DEMO)\Final
Processing FinalOutcome...
Missing files in D:\MONITOR(DEMO)\FinalOutcome
All subjects processed!

=== MODEL RESULT ===
Weights (scaled): [0.02880185 0.0833638  0.01001935]
Intercept: 1.664457408537914
R² Score: 0.1556304380431488

=== FINAL EQUATION (RAW DATA) ===
Stress = 0.001867692637527866*HR + 0.0007349479947013937*GSR + 0.0018291312194508222*Temp + 1.1882542615212734
